# Stellarator isodrasticity — Julia notebook



In [34]:
# Point at the repo root (edit if your notebook lives elsewhere)
const REPO = raw"C:/Users/danie/Stellarator_Isodrasticity"
cd(REPO)
pwd()

"C:\\Users\\danie\\Stellarator_Isodrasticity"

## Load `magnetic_ridge_3` output (JLD2)

Keys are typically `ridge_x`, `ridge_y`, `ridge_z` and full grid `grid_x`, `grid_y`, `grid_z` (`NaN` where no ridge).

In [35]:
using JLD2

ridge_jld2 = joinpath(REPO, "data", "magnetic_ridge3", "magnetic_ridge_3_1.jld2")

rx, ry, rz, meta = jldopen(ridge_jld2, "r") do f
    Vector{Float64}(f["ridge_x"]),
    Vector{Float64}(f["ridge_y"]),
    Vector{Float64}(f["ridge_z"]),
    haskey(f, "meta") ? f["meta"] : nothing
end

println("Ridge points: ", length(rz))
meta

Ridge points: 9368


Dict{String, Any} with 10 entries:
  "nx"               => 200
  "multi_ridge_note" => "multi_ridge_*: only rows where >1 valid ridge root; va…
  "output_jld2_base" => "c:\\Users\\danie\\Stellarator_Isodrasticity\\data\\mag…
  "z_pad"            => 0.01
  "conf_path"        => "C:/Users/danie/Stellarator_Isodrasticity/data/confinem…
  "output_jld2_path" => "c:\\Users\\danie\\Stellarator_Isodrasticity\\data\\mag…
  "ny"               => 200
  "min_neighbors"    => 18
  "xy_radius_base"   => 0.06
  "xy_radius_max"    => 0.25

## Quick 3D scatter 

Blue for main values, gold for duplicate

In [39]:
using CairoMakie

R = sqrt.(rx .^ 2 .+ ry .^ 2)
fig = Figure(size = (800, 600))
ax = Axis3(fig[1, 1]; title = "Ridge points", aspect = :data)
scatter!(ax, rx, ry, rz; color = :blue,  markersize = 8)

#Load duplicates

mux, muy, muzs = jldopen(ridge_jld2, "r") do f
    if !haskey(f, "multi_ridge_xy_x")
        println("No multi_ridge_* in this file")
        nothing, nothing, nothing
    else
        Vector{Float64}(f["multi_ridge_xy_x"]),
        Vector{Float64}(f["multi_ridge_xy_y"]),
        f["multi_ridge_valid_zs"]
    end
end


    
    
scatter!(ax, mx_all, my_all, mz_all; color = :gold, markersize = 10)
    

fig

## K-means clustering (3D)

**k-means++** initialization plus **Lloyd iterations**: assign each ridge point to the nearest centroid in *normalized* \((x,y,z)\) space (z-score so no axis dominates), update centroids as cluster means, repeat until relative SSE improvement \(<\) tolerance.

Same logic as `cluster_magnetic_ridge_v3.jl` (without saving JLD2 here).

In [30]:
using Statistics
using Random
using LinearAlgebra

@inline sqdist3(x1, y1, z1, x2, y2, z2) = (x1 - x2)^2 + (y1 - y2)^2 + (z1 - z2)^2

function kmeanspp_init(x::Vector{Float64}, y::Vector{Float64}, z::Vector{Float64}, k::Int, rng::AbstractRNG)
    n = length(x)
    cidx = Vector{Int}(undef, k)
    cidx[1] = rand(rng, 1:n)
    d2 = fill(Inf, n)
    for j in 2:k
        cj = cidx[j - 1]
        @inbounds for i in 1:n
            dij = sqdist3(x[i], y[i], z[i], x[cj], y[cj], z[cj])
            if dij < d2[i]
                d2[i] = dij
            end
        end
        s = sum(d2)
        if s <= 0
            cidx[j] = rand(rng, 1:n)
            continue
        end
        r = rand(rng) * s
        acc = 0.0
        chosen = n
        @inbounds for i in 1:n
            acc += d2[i]
            if acc >= r
                chosen = i
                break
            end
        end
        cidx[j] = chosen
    end
    return cidx
end

"""Lloyd k-means in weighted z-score space; `z_weight<1` emphasizes XY. Returns labels and centroids in original coordinates."""
function run_kmeans3(
    x::Vector{Float64},
    y::Vector{Float64},
    z::Vector{Float64},
    k::Int;
    maxiter::Int = 200,
    tol::Float64 = 1e-6,
    seed::Int = 1234,
    z_weight::Float64 = 0.2,
)
    n = length(x)
    @assert 1 <= k <= n
    μx, μy, μz = mean(x), mean(y), mean(z)
    σx, σy, σz = std(x), std(y), std(z)
    σx = σx == 0 ? 1.0 : σx
    σy = σy == 0 ? 1.0 : σy
    σz = σz == 0 ? 1.0 : σz
    xn = (x .- μx) ./ σx
    yn = (y .- μy) ./ σy
    z_weight_safe = max(z_weight, 1e-12)
    zn = ((z .- μz) ./ σz) .* z_weight_safe
    rng = MersenneTwister(seed)
    cidx = kmeanspp_init(xn, yn, zn, k, rng)
    cx = xn[cidx]; cy = yn[cidx]; cz = zn[cidx]
    labels = zeros(Int, n)
    prev_sse = Inf
    for _ in 1:maxiter
        sse = 0.0
        @inbounds for i in 1:n
            bestj = 1
            bestd = sqdist3(xn[i], yn[i], zn[i], cx[1], cy[1], cz[1])
            for j in 2:k
                d = sqdist3(xn[i], yn[i], zn[i], cx[j], cy[j], cz[j])
                if d < bestd
                    bestd = d
                    bestj = j
                end
            end
            labels[i] = bestj
            sse += bestd
        end
        sumx = zeros(k); sumy = zeros(k); sumz = zeros(k); cnt = zeros(Int, k)
        @inbounds for i in 1:n
            j = labels[i]
            sumx[j] += xn[i]; sumy[j] += yn[i]; sumz[j] += zn[i]
            cnt[j] += 1
        end
        for j in 1:k
            if cnt[j] == 0
                r = rand(rng, 1:n)
                cx[j] = xn[r]; cy[j] = yn[r]; cz[j] = zn[r]
            else
                cx[j] = sumx[j] / cnt[j]
                cy[j] = sumy[j] / cnt[j]
                cz[j] = sumz[j] / cnt[j]
            end
        end
        rel = abs(prev_sse - sse) / max(prev_sse, 1.0)
        if rel < tol
            break
        end
        prev_sse = sse
    end
    cx0 = cx .* σx .+ μx
    cy0 = cy .* σy .+ μy
    cz0 = (cz ./ z_weight_safe) .* σz .+ μz
    return labels, cx0, cy0, cz0
end

function split_clusters(x::Vector{Float64}, y::Vector{Float64}, z::Vector{Float64}, labels::Vector{Int}, k::Int)
    xs = [Float64[] for _ in 1:k]
    ys = [Float64[] for _ in 1:k]
    zs = [Float64[] for _ in 1:k]
    @inbounds for i in eachindex(labels)
        j = labels[i]
        push!(xs[j], x[i]); push!(ys[j], y[i]); push!(zs[j], z[i])
    end
    return xs, ys, zs
end

split_clusters (generic function with 1 method)

### Run k-means on `magnetic_ridge_3` points

Set **`K`** = number of discrete segments (clusters). Re-run the data cell above first so `rx`, `ry`, `rz` are defined (or reload from JLD2 here).

In [31]:
# Number of discrete ridge segments (clusters) — change and re-run this cell
K = 10

n = length(rz)
@assert K >= 1 && K <= n

KMEANS_Z_WEIGHT = 0.2
labels, ctr_x, ctr_y, ctr_z = run_kmeans3(rx, ry, rz, K; maxiter = 200, tol = 1e-6, seed = 1234, z_weight = KMEANS_Z_WEIGHT)
cluster_xs, cluster_ys, cluster_zs = split_clusters(rx, ry, rz, labels, K)
sizes = [length(cluster_xs[j]) for j in 1:K]
println("Cluster sizes: ", sizes)
(labels, ctr_x, ctr_y, ctr_z)

Cluster sizes: [1613, 1584, 607, 539, 718, 1239, 748, 674, 751, 895]


([7, 7, 4, 4, 4, 7, 7, 4, 4, 4  …  8, 8, 8, 8, 8, 8, 1, 1, 8, 8], [0.7424584299939657, 0.6950860041945285, -0.8698772460982565, -1.1822670000408313, 0.007797105917047685, -0.6467176406856395, -0.9482265417830255, 1.1636556658796662, -0.01698167989225674, -0.5161250990624705], [-0.588858626164023, 0.631734392948761, 0.24283196666232457, 0.04588639764573854, -0.8645457113706072, 0.6921712135542887, -0.448751966196585, -0.05628979553084966, 0.8509391612628066, -0.7552102043102643], [-0.1453971415511068, 0.15095454592890747, -0.2353143019100824, -0.028198601550011707, 0.002835206737367053, -0.13022667703004576, 0.16077083588030514, 0.06496920245912008, -0.000772682973908131, 0.12742114454126602])

### Plot ridge points colored by cluster

Uses a discrete palette so each of the **K** segments has a distinct color; centroids are marked with black diamonds.

In [32]:
using GLMakie

pal = Makie.wong_colors()
pt_colors = [pal[mod1(labels[i], length(pal))] for i in eachindex(labels)]

fig_k = Figure(size = (900, 700))
ax_k = Axis3(fig_k[1, 1]; title = "Magnetic ridge colored by k-means cluster (K=$(K))", aspect = :data)
scatter!(ax_k, rx, ry, rz; color = pt_colors, markersize = 10, transparency = false)
scatter!(ax_k, ctr_x, ctr_y, ctr_z; color = :black, marker = :diamond, markersize = 18)
fig_k

### DBSCAN (density-based clustering)

Implementation: **`Clustering.dbscan`** ([Clustering.jl](https://juliastats.org/Clustering.jl)) with a **KD-tree** neighborhood search on **z-scored** \((x,y,z)\) coordinates (same scale as k-means). **DBSCAN** assigns **core / border / noise** points; **noise** is label `0`, clusters are `1, 2, …` (number of clusters is **not** fixed in advance).

Tune **`db_eps`** (radius) and **`db_minpts`** (minimum neighbors, matching `Clustering`’s `min_neighbors`). Larger `eps` → larger neighborhoods; larger `minpts` → stricter cores.

In [33]:
try
    using Clustering
catch
    import Pkg
    Pkg.add("Clustering")
    using Clustering
end
using Statistics

# Clustering.dbscan expects d×n columns = points; use z-scored (x,y,z) like k-means
xn, yn, zn = let
    μx, μy, μz = mean(rx), mean(ry), mean(rz)
    σx, σy, σz = std(rx), std(ry), std(rz)
    σx = σx == 0 ? 1.0 : σx
    σy = σy == 0 ? 1.0 : σy
    σz = σz == 0 ? 1.0 : σz
    ((rx .- μx) ./ σx, (ry .- μy) ./ σy, (rz .- μz) ./ σz)
end
DBSCAN_Z_WEIGHT = 0.2
X = vcat(xn', yn', (DBSCAN_Z_WEIGHT .* zn)')

db_eps = 0.35
db_minpts = 12

res_db = dbscan(X, db_eps; min_neighbors = db_minpts, min_cluster_size = 1)
labels_db = res_db.assignments
n_clu = maximum(labels_db)
n_noise = count(==(0), labels_db)
println("DBSCAN (Clustering.jl): clusters = ", n_clu, " | noise = ", n_noise, " | eps=", db_eps, " minpts=", db_minpts, " | z_weight=", DBSCAN_Z_WEIGHT)

pal_db = Makie.wong_colors()
db_colors = [
    labels_db[i] == 0 ? RGBf(0.55, 0.55, 0.58) :
    pal_db[mod1(labels_db[i], length(pal_db))] for i in eachindex(labels_db)
]

fig_db = Figure(size = (900, 700))
ax_db = Axis3(
    fig_db[1, 1];
    title = "DBSCAN (noise gray; ε=$(db_eps), minpts=$(db_minpts))",
    aspect = :data,
)
scatter!(ax_db, rx, ry, rz; color = db_colors, markersize = 8)
fig_db


DBSCAN: clusters = 8 | noise points = 17 | eps=0.35 minpts=12


### Spectral clustering (normalized graph Laplacian)

**Idea:** build a **similarity graph** on a **subsample** of points (full \(n\) is too large for a dense \(n\times n\) affinity). Use a **Gaussian kernel** \(W_{ij}=\exp(-\|\mathbf{x}_i-\mathbf{x}_j\|^2/(2\sigma^2))\) with an **XY-dominant distance** (z-scored \((x,y,z)\) but with configurable **`SPEC_Z_WEIGHT < 1`** on \(z\)). Form the **symmetric normalized Laplacian** \(L_{\mathrm{sym}} = I - D^{-1/2} W D^{-1/2}\) (zero diagonal on \(W\)). Take the **first \(K\) nontrivial** eigenvectors (columns after the constant mode), **row-normalize** them, run **k-means** (`Clustering.kmeans`) in that embedding, then **copy labels** to all points by nearest neighbor in the same weighted space.

Tune **`K_spec`**, **`m_sub`**, and **`SPEC_Z_WEIGHT`** (smaller means more XY-driven clustering).


In [ ]:
using LinearAlgebra
using Random
using Statistics
try
    using Clustering
catch
    import Pkg
    Pkg.add("Clustering")
    using Clustering
end

K_spec = 10
m_sub = 4000
seed_sp = 42
SPEC_Z_WEIGHT = 0.2

n = length(rx)
m_sub = min(m_sub, n)
K_spec = max(2, min(K_spec, m_sub - 2))

rng = MersenneTwister(seed_sp)
sub = sort(randperm(rng, n)[1:m_sub])
in_sub = falses(n)
in_sub[sub] .= true

xn, yn, zn = let
    μx, μy, μz = mean(rx), mean(ry), mean(rz)
    σx, σy, σz = std(rx), std(ry), std(rz)
    σx = σx == 0 ? 1.0 : σx
    σy = σy == 0 ? 1.0 : σy
    σz = σz == 0 ? 1.0 : σz
    ((rx .- μx) ./ σx, (ry .- μy) ./ σy, (rz .- μz) ./ σz)
end

Psub = zeros(m_sub, 3)
for (s, idx) in enumerate(sub)
    Psub[s, 1] = xn[idx]
    Psub[s, 2] = yn[idx]
    Psub[s, 3] = SPEC_Z_WEIGHT * zn[idx]
end

D = zeros(m_sub, m_sub)
@inbounds for i in 1:m_sub
    for j in i+1:m_sub
        d2 =
            (Psub[i, 1] - Psub[j, 1])^2 +
            (Psub[i, 2] - Psub[j, 2])^2 +
            (Psub[i, 3] - Psub[j, 3])^2
        dij = sqrt(d2)
        D[i, j] = D[j, i] = dij
    end
end

dv = Float64[]
for i in 1:m_sub, j in i+1:m_sub
    push!(dv, D[i, j])
end
σ_sp = max(median(dv), eps(Float64))

W = exp.(-(D .^ 2) ./ (2 * σ_sp^2))
for i in 1:m_sub
    W[i, i] = 0.0
end

dvec = vec(sum(W, dims = 2))
dvec = max.(dvec, 1e-12)
Dmh = Diagonal(1 ./ sqrt.(dvec))
Lsym = I(m_sub) - Dmh * W * Dmh
Lsym = Symmetric((Lsym + Lsym') / 2)

F = eigen(Lsym)
perm = sortperm(F.values)
vecs = F.vectors[:, perm]
U = vecs[:, 2:K_spec+1]

for i in 1:m_sub
    r = norm(U[i, :])
    if r > 0
        U[i, :] ./= r
    end
end

km_sp = kmeans(permutedims(U), K_spec; maxiter = 300)
assign_sub = km_sp.assignments

labels_sp = zeros(Int, n)
for s in 1:m_sub
    labels_sp[sub[s]] = assign_sub[s]
end

for i in 1:n
    in_sub[i] && continue
    best_j = sub[1]
    best_d = Inf
    for j in sub
        dd = (xn[i] - xn[j])^2 + (yn[i] - yn[j])^2 + (SPEC_Z_WEIGHT * (zn[i] - zn[j]))^2
        if dd < best_d
            best_d = dd
            best_j = j
        end
    end
    labels_sp[i] = labels_sp[best_j]
end

n_sp = maximum(labels_sp)
println("Spectral: K_spec=", K_spec, " m_sub=", m_sub, " σ=", round(σ_sp, sigdigits = 5), " z_weight=", SPEC_Z_WEIGHT, " max label=", n_sp)

pal_sp = Makie.wong_colors()
sp_colors = [pal_sp[mod1(labels_sp[i], length(pal_sp))] for i in eachindex(labels_sp)]

fig_sp = Figure(size = (900, 700))
ax_sp = Axis3(
    fig_sp[1, 1];
    title = "Spectral clustering (K_spec=$(K_spec), m_sub=$(m_sub), z_w=$(SPEC_Z_WEIGHT))",
    aspect = :data,
)
scatter!(ax_sp, rx, ry, rz; color = sp_colors, markersize = 8)
fig_sp


### Per-cluster optimization (polynomial–Fourier least squares)

Same model as `parameterize_ridge_v3.jl`:

\[
z(R,\phi) = \sum_{m=0}^{M} \sum_{n=0}^{N} R^m \bigl( A_{m,n}\cos(n\phi) + B_{m,n}\sin(n\phi) \bigr)
\]

with \(n=0\) cosine-only. Coefficients are found by **linear least squares** `c = A \\ z` per cluster.

In [ ]:
using Printf

function n_coefficients(M::Int, N::Int)
    (M + 1) + N * (M + 1) * 2
end

function design_matrix(R::AbstractVector{Float64}, phi::AbstractVector{Float64}, M::Int, N::Int)
    npts = length(R)
    @assert length(phi) == npts
    ncols = n_coefficients(M, N)
    A = zeros(npts, ncols)
    col = 1
    @inbounds for m in 0:M
        A[:, col] .= R .^ m
        col += 1
    end
    for n in 1:N
        @inbounds for m in 0:M
            A[:, col] .= (R .^ m) .* cos.(n .* phi)
            col += 1
            A[:, col] .= (R .^ m) .* sin.(n .* phi)
            col += 1
        end
    end
    @assert col == ncols + 1
    return A
end

function predict_z(R::AbstractArray{Float64}, phi::AbstractArray{Float64}, c::AbstractVector{Float64}, M::Int, N::Int)
    z = zeros(Float64, size(R))
    k = 1
    @inbounds for m in 0:M
        z .+= c[k] .* (R .^ m)
        k += 1
    end
    for n in 1:N
        @inbounds for m in 0:M
            z .+= c[k] .* ((R .^ m) .* cos.(n .* phi))
            k += 1
            z .+= c[k] .* ((R .^ m) .* sin.(n .* phi))
            k += 1
        end
    end
    return z
end

# Match parameterize_ridge_v3 defaults unless you change these
M_rad = 2
N_tor = 3

cluster_cs = Vector{Vector{Float64}}(undef, K)
cluster_rmses = Vector{Float64}(undef, K)

for j in 1:K
    xv = cluster_xs[j]
    yv = cluster_ys[j]
    zv = cluster_zs[j]
    if isempty(zv)
        cluster_cs[j] = Float64[]
        cluster_rmses[j] = NaN
        @warn "Cluster $(j) is empty"
        continue
    end
    Rc = hypot.(xv, yv)
    phic = atan.(yv, xv)
    A = design_matrix(Rc, phic, M_rad, N_tor)
    cj = A \ zv
    zh = A * cj
    cluster_cs[j] = cj
    cluster_rmses[j] = sqrt(sum(abs2, zv .- zh) / length(zv))
    println(@sprintf("Cluster %d: n=%d  RMSE=%.6g", j, length(zv), cluster_rmses[j]))
end

(cluster_cs, cluster_rmses)

### Plot each cluster: data + fitted surface

One 3D panel per cluster; scatter matches the cluster color; translucent surface is \(z(R,\phi)\) from the fit on that segment’s \((R,\phi)\) range.

In [ ]:
using Printf

SURF_NR = 60
SURF_NPHI = 90
pal = Makie.wong_colors()
ncols = min(3, K)
nrows = cld(K, ncols)
fig_fit = Figure(size = (380 * ncols + 80, 360 * nrows + 40))

for j in 1:K
    r = div(j - 1, ncols) + 1
    icol = mod(j - 1, ncols) + 1
    ax = Axis3(
        fig_fit[r, icol];
        title = @sprintf("Cluster %d  RMSE=%.4g", j, cluster_rmses[j]),
        aspect = :data,
    )
    xv = cluster_xs[j]
    yv = cluster_ys[j]
    zv = cluster_zs[j]
    isempty(xv) && continue
    cj = cluster_cs[j]
    isempty(cj) && continue
    clr = pal[mod1(j, length(pal))]
    scatter!(ax, xv, yv, zv; color = clr, markersize = 8)
    Rc = hypot.(xv, yv)
    Rmin = minimum(Rc)
    Rmax = maximum(Rc)
    Rg = range(Rmin, Rmax; length = SURF_NR)
    phig = range(-Float64(π), Float64(π); length = SURF_NPHI)
    Rmat = [rr for rr in Rg, _ in phig]
    phimat = [p for _ in Rg, p in phig]
    Zmat = predict_z(Rmat, phimat, cj, M_rad, N_tor)
    Xmat = Rmat .* cos.(phimat)
    Ymat = Rmat .* sin.(phimat)
    surface!(ax, Xmat, Ymat, Zmat; colormap = :viridis, alpha = 0.55, shading = true)
end

fig_fit